# VoiceNO: measured silent-speech baseline

Use your own interactive Colab GPU session. No public server or paid API is started.

Upload the current source bundle prepared on your laptop, then upload existing recordings or record new clips here. Uploaded videos and webcam clips are sent to this remote runtime. No microphone is requested. Do not record sensitive content for this first experiment.

This notebook measures predictions; it does not establish that the model is accurate, commercially cleared, or running inside an enclave. Start with the generic baseline. Personalization is optional after a separate calibration step.


In [ ]:
# 1. Verify a GPU before installing/loading the recognition model.
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime > Change runtime type > GPU, then reconnect. Free GPUs may be unavailable.")
print(torch.cuda.get_device_name(0))


## Upload the current source

On your laptop run `python experiments/package_eval_source.py`, then select `voiceno_eval_source.zip` below. This archive excludes your recordings and model/profile weights. It includes local changes that may not yet be on GitHub.


In [ ]:
# 2. Upload and verify the source archive.
from google.colab import files
import hashlib
import io
import json
import os
from pathlib import Path
import subprocess
import sys
import tempfile
import zipfile

SOURCE_ROOT = Path(tempfile.mkdtemp(prefix="voiceno-eval-", dir="/content"))
def extract_checked(payload, destination):
    destination = Path(destination).resolve()
    with zipfile.ZipFile(io.BytesIO(payload)) as archive:
        if sum(item.file_size for item in archive.infolist()) > 500 * 1024 * 1024:
            raise ValueError("Archive is too large for this small baseline workflow")
        for item in archive.infolist():
            if not (destination / item.filename).resolve().is_relative_to(destination):
                raise ValueError("Archive contains a path outside the destination")
        archive.extractall(destination)

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Select just voiceno_eval_source.zip")
extract_checked(next(iter(uploaded.values())), SOURCE_ROOT)
source_manifest = json.loads((SOURCE_ROOT / "evaluation_source_manifest.json").read_text())
for name, expected in source_manifest["files"].items():
    path = (SOURCE_ROOT / name).resolve()
    if not path.is_relative_to(SOURCE_ROOT):
        raise ValueError("Invalid manifest path")
    if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
        raise ValueError(f"Source hash mismatch: {name}")
del uploaded
os.chdir(SOURCE_ROOT)
sys.path.insert(0, str(SOURCE_ROOT))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("Source extracted and verified:", SOURCE_ROOT)


In [ ]:
# 3. Download the existing checkpoint and verify its documented checksum.
import gdown
checkpoint = SOURCE_ROOT / "checkpoints/vsr_trlrs2lrs3vox2avsp_base.pth"
checkpoint.parent.mkdir(exist_ok=True)
if not checkpoint.exists():
    gdown.download(id="1r1kx7l9sWnDOCnaFHIGvOtzuhFyFA88_", output=str(checkpoint), quiet=False)
checksum = hashlib.md5()
with checkpoint.open("rb") as stream:
    for chunk in iter(lambda: stream.read(1024 * 1024), b""):
        checksum.update(chunk)
if checksum.hexdigest() != "49f770f2c0d8b8d769347ee47ed1648f":
    raise ValueError("Checkpoint checksum mismatch; do not evaluate these weights")
print("Checkpoint checksum matched. Model/data usage rights remain a separate question.")


## Supply real evaluation clips

Choose a user ID below. If you have recordings, export them locally with `python -m voiceno export-calib --user default` and upload that archive. It contains video. Review its reference labels and ensure evaluation recordings were not used for training.

Alternatively skip the upload and use the following optional camera cell repeatedly. Start with 10–20 short, genuinely silent AI prompts of your own. Do not reuse calibration clips. One clip only establishes that the pipeline runs.


In [ ]:
# 4. Optional: upload your existing calibration/evaluation archive.
USER_ID = "default"
UPLOAD_EXISTING = True  # Set False if recording new clips below instead.
if UPLOAD_EXISTING:
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Select one calibration package")
    extract_checked(next(iter(uploaded.values())), SOURCE_ROOT / "calibration_data" / USER_ID)
    del uploaded
print("Evaluation directory:", SOURCE_ROOT / "calibration_data" / USER_ID / "evaluation")


In [ ]:
# 5. Optional: edit the prompt and rerun once for each NEW evaluation clip.
RECORD_A_NEW_CLIP = False
PROMPT = "Explain why this function returns an empty list"
if RECORD_A_NEW_CLIP:
    import cv2
    import uuid
    from app.colab_recorder import record_webcam
    from voiceno.calibration.recorder import CalibrationRecorder
    capture_path = SOURCE_ROOT / f"capture-{uuid.uuid4().hex}.mp4"
    video_path = record_webcam(str(capture_path))
    capture = cv2.VideoCapture(video_path)
    frames = []
    while True:
        ok, frame = capture.read()
        if not ok:
            break
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    capture.release()
    if len(frames) < 5:
        raise ValueError("Recording has too few readable frames; please record again")
    saved = CalibrationRecorder().save_utterance(USER_ID, frames, PROMPT, partition="evaluation", fps=25.0)
    capture_path.unlink(missing_ok=True)
    (SOURCE_ROOT / "temp_webcam.webm").unlink(missing_ok=True)
    print("Saved:", saved["utterance_id"], saved["prompt"], saved["duration_seconds"], "seconds")
    print("Confirm you mouthed this exact reference. Delete/rerecord an incorrect attempt before evaluation.")


In [ ]:
# 6. Run the generic baseline on the remote GPU. No training/profile required.
from experiments.run_open_vocab_eval import run_open_vocab_evaluation
report = run_open_vocab_evaluation(USER_ID, device="cuda", beam_size=10)
report["provenance"]["uploaded_source_manifest"] = source_manifest
REPORT_PATH = SOURCE_ROOT / "experiments" / f"open_vocab_benchmark_{USER_ID}.json"
REPORT_PATH.write_text(json.dumps(report, indent=2))
for row in report["conditions"]["A_generic_baseline"]["utterances"]:
    print("Reference:", row["reference"])
    print("Prediction:", row["hypothesis"])
    print("Status:", row["status"], "WER:", row["wer"], "Error:", row["error"])
    print()


## Optional later comparison

After training a profile on separate calibration clips, enable the next cell and upload its `lora.pt` and `metadata.json`. This compares actual generic/personalized weights and reuses the personalized candidates for context rescoring. Oracle scores use reference answers; they are candidate-recovery ceilings. Context scoring currently uses hand-written heuristics.


In [ ]:
# 7. Optional: compare a separately trained profile.
COMPARE_PERSONALIZED = False
if COMPARE_PERSONALIZED:
    profile_dir = SOURCE_ROOT / "user_profiles" / USER_ID
    profile_dir.mkdir(parents=True, exist_ok=True)
    uploaded = files.upload()
    if set(uploaded) != {"lora.pt", "metadata.json"}:
        raise ValueError("Select exactly lora.pt and metadata.json from your trained profile")
    for name, content in uploaded.items():
        (profile_dir / name).write_bytes(content)
    del uploaded
    report = run_open_vocab_evaluation(USER_ID, personalized=True, device="cuda", beam_size=10)
    report["provenance"]["uploaded_source_manifest"] = source_manifest
    REPORT_PATH.write_text(json.dumps(report, indent=2))


In [ ]:
# 8. Download actual predictions for inspection.
files.download(str(REPORT_PATH))


When finished, clear saved notebook outputs containing reference text/predictions and remove uploaded recordings from the runtime. Downloaded reports and any exported videos remain where you saved them. No secure-erasure guarantee is made. Free GPU interruptions and package compatibility need testing in your actual Colab session; this notebook has been checked locally for structure, not executed on a remote GPU.
